# 추천 시스템

## 추천 시스템 기초

> ### 추천 시스템과 데이터 유형

>> 1) 추천 시스템이란?
- 사용자의 행동 기록과 아이템 정보를 바탕으로 **좋아할 만한 아이템을 예측해 골라 주는 시스템**
    - 아이템: 상품, 영화, 음악, 뉴스, 앱 등
    - 예: 넷플릭스 추천 콘텐츠, 유튜브 다음 영상, 쇼핑몰 "함께 본 상품"
- **필요한 이유**
    - 고객 입장
        - 찾는 시간과 노력 감소
        - 몰랐던 취향의 아이템 발견
        - **검색**은 아는 것을 찾게 돕고, **추천**은 모르는 것을 발견하게 도움
    - 기업 입장
        - 이용 시간·구매 전환·재방문 증가
        - 비인기 아이템(**롱테일**) 노출

>> 2) 데이터 유형
- **명시적 데이터(Explicit)**
    - 사용자가 선호를 **직접 표현**한 데이터 (별점, 좋아요·싫어요, 리뷰)
    - 장점: 선호·불호가 분명해 취향을 정확히 반영
    - 단점
        - 평가를 남기는 사람이 적고, 본 것 중 일부에만 남김
        - 취향은 변하므로 과거 선호가 현재와 같은지 알 수 없음
        - 사람마다 점수 기준이 다름
- **암시적 데이터(Implicit)**
    - 행동에서 선호를 **간접적으로 추측**하는 데이터 (조회수, 클릭, 구매, 시청 시간, 검색)
    - 장점: 따로 입력하지 않아도 대량으로 쌓임
    - 단점
        - 행동이 곧 선호라고 단정할 수 없음 (잘못된 클릭, 선물 구매)
        - 행동이 없어도 싫어한다는 보장이 없음 (아직 못 봤을 수 있음)
- 실제 서비스는 암시적 데이터가 훨씬 많아 **두 가지를 혼합**해 사용

> ### 추천 시스템 분류

>> 1) 내용 기반 필터링
- 아이템 **유사도**를 근거로 추천
- 사용자가 선택한 아이템과 **특징이 비슷한 아이템** 추천
    - 예: SF 영화를 자주 봄 → 다른 SF 영화 추천

>> 2) 협업 필터링
- **다수 사용자의 선호도 정보**를 근거로 추천
- 취향이 비슷한 사용자는 한 사람이 좋아한 아이템을 다른 사람도 좋아할 가능성이 높다고 가정
- **사용자 기반**: 유사한 취향의 사용자가 선호한 아이템 추천
- **아이템 기반**: 사용자가 선호한 아이템과 유사한 아이템 추천

>> 3) 하이브리드 필터링
- 2가지 이상의 알고리즘을 **조합**해 각각의 단점을 보완
    - 예: 평점 없는 신작은 내용 기반, 평점이 쌓이면 협업 필터링
- 하이브리드 방식
    - **Weighted Ensemble**: 여러 모델의 추천 결과를 합쳐 최종 아이템 결정
    - **Mixed**: 특징이 다른 여러 알고리즘의 결과를 함께 노출
    - **Switch**: 사용자·서비스 상태에 따라 알고리즘 선택 (신작 → 내용 기반, 평점 축적 → 협업 필터링)
    - **Feature Combination**: 다양한 Feature를 조합해 하나의 알고리즘으로 학습
    - **Meta-Level**: 첫 번째 모델의 출력이 다음 모델의 입력

---

## 내용 기반 필터링

> ### 사용자 프로필과 유사도
- 핵심 흐름: **아이템을 숫자로 표현 → 사용자 프로필 생성 → 유사도 계산 → 점수 높은 아이템 추천**

>> 1) 사용자 프로필
- 사용자가 어떤 특성을 좋아하는지 정리한 정보
- 획득 방식
    - **직접 지정형**: 가입 시 좋아하는 장르·아티스트 선택
    - **간접 지정형**: 과거 행동 이력에서 취향 추론
- 구성 방법
    - **아이템 특성 가중치의 평균**
        - 사용자 프로필 = 본 영화들의 특성 벡터 평균
        - 예: 어벤져스 `[0.9, 0.1, 0.5, 0.0]`, 인터스텔라 `[0.3, 0.2, 0.9, 0.0]`, 노트북 `[0.0, 0.9, 0.0, 0.1]` → `[0.4, 0.4, 0.47, 0.03]`
        - 신규 영화(듄 `[0.4, 0.0, 0.9, 0.0]`)와 코사인 유사도 약 0.8 → 추천
    - **설문·키워드·평가로 기호 파악**
        - 명시적: 좋아요/싫어요, 평점, 댓글
        - 암시적: 검색 로그, 구매, 저장, 북마크
    - **분류(Classification) 문제로 해결**
        - 아이템 특징이 입력, 유저 평가가 정답
        - 결정 트리, 랜덤 포레스트, KNN 적용 가능

>> 2) 유사도(Similarity)
- 두 대상이 얼마나 비슷한지 나타내는 숫자
- 사용자와 아이템을 **같은 feature 기준**으로 표현해야 비교 가능
- **STEP 1. 아이템을 숫자로 변환**
    - 장르·국가·감독·배우 등 속성 = **feature**
    - 행: 영화 제목 / 열: feature / 셀: feature 값
    - **원-핫 인코딩**: 속하면 1, 아니면 0 (예: `[1, 0, 0, 1, ...]`)
- **STEP 2. 사용자를 숫자로 변환**
    - 직접 입력한 프로필 + 시청 기록
    - 아이템과 **동일한 n개 feature**로 표현
- **STEP 3. 유사도 계산**
    - **벡터 내적**: 동일 위치 값끼리 곱해 합산, 클수록 유사
    - 0/1 벡터의 내적 = **겹치는 feature 개수**
- 유사도 종류
    - **자카드**: 집합 간 교집합 크기 기반
    - **코사인**: 벡터 사이 각도 기반
    - **유클리디안 거리**: 벡터 사이 거리 기반
    - **피어슨 상관계수**: 벡터 간 선형 상관 기반
- **코사인 유사도**
    - 각도가 작을수록 1에 가까움, 값이 클수록 비슷

$$
\cos(A, B) = \cos\theta = \frac{A \cdot B}{\|A\| \, \|B\|}
$$

    - 길이는 무시하고 **방향**만 비교 (`[1, 1]`과 `[5, 5]`는 유사도 1)
    - 거리로 쓰려면 **코사인 거리** = 1 − 코사인 유사도
- **내적 vs 코사인**
    - 내적: 길이를 반영해 feature가 많은 아이템(장르 여러 개)이 유리
    - 코사인: feature 개수로 인한 쏠림은 줄지만 정보의 풍부함은 반영 못 함

>> 3) TF-IDF
- 텍스트(줄거리·상품 설명)를 숫자로 바꿀 때 사용
- 단순 단어 빈도는 "그리고", "있다" 같은 흔한 단어를 중요하게 취급하는 문제
- **특정 문서에는 자주, 다른 문서에는 드문 단어**에 높은 값 부여
    - **TF**: 문서에 자주 나올수록 커짐
    - **IDF**: 등장한 문서 수가 많을수록 작아짐

$$
idf(d,t) = \log\left(\frac{n}{1+df(t)}\right), \quad TF\text{-}IDF = TF(d,t) \times IDF(d,t)
$$

- 예시 (문서 A "고양이는 귀엽다", B "강아지는 귀엽다", C "고양이는 사냥한다", n=3)
    - "귀엽다": DF=2 → IDF = log(1) = 0 → TF-IDF 0 (중요도 낮음)
    - "사냥한다": DF=1 → IDF = log(1.5) ≈ 0.176 → TF-IDF 0.176 (상대적으로 중요)
- 추천 적용
    1. 영화 줄거리를 **TF-IDF 벡터**로 변환
    2. 본 영화 벡터와 다른 영화를 **코사인 유사도**로 비교
    3. 유사도 높은 영화 추천
- 코드
    - `TfidfVectorizer`로 벡터화 → `cosine_similarity`로 유사도 계산

> ### 장단점

>> 1) 장점
- **아이템 콜드 스타트 해결**: 속성만 있으면 신규 아이템 추천 가능
- 내 기록만으로 프로필을 만들어 **다른 사용자 데이터 불필요**
- feature 기반이라 **추천 이유 설명 용이**

>> 2) 단점
- feature로 담을 수 있는 정보에 한계, 속성 부족·저품질 시 **정확도 하락**
- **과도한 특화(Over-specialization)**: 비슷한 것만 계속 추천
- **사용자 콜드 스타트**: 기록 없는 신규 사용자는 추천 어려움

>> 3) 콜드 스타트 문제
- 충분한 정보가 없어 적절한 추천을 못 하는 문제
    - **아이템 콜드 스타트**: 내용 기반으로 해결 가능
    - **사용자 콜드 스타트**: 내용 기반으로도 해결 어려움

---

## 협업 필터링

> 가정: 과거에 비슷한 선호도를 가진 사용자들은 앞으로도 비슷한 선호도를 가질 가능성이 높다.

> ### 메모리 기반 협업 필터링

>> 1) 개념
- 저장된 사용자-아이템 평점을 **그대로 꺼내** 새 아이템 평점 예측
- 별도 학습 없이 비슷한 사용자·아이템, 즉 **이웃**을 찾아 사용 → **이웃 기반 CF**
- 협업 필터링 중 비교적 초기 방식

>> 2) 사용자 기반(User-based)
- 취향이 비슷한 사용자(이웃)의 평점으로 내 평점 예측
    - 유사도 상위 `TOP-N`명을 이웃으로 선정
    - 이웃 평점을 **유사도로 가중평균**
    - 예측 평점이 높은 아이템 추천
- 사용자 간 유사도: **피어슨 상관계수**

$$
sim(u, v) = \frac{\sum_{i} (r_{ui} - \bar{r}_u)(r_{vi} - \bar{r}_v)}{\sqrt{\sum_{i} (r_{ui} - \bar{r}_u)^2} \sqrt{\sum_{i} (r_{vi} - \bar{r}_v)^2}}
$$

    - $r_{ui}$: 사용자 $u$가 아이템 $i$에 준 평점, $\bar{r}_u$: $u$의 평균 평점, $i$: 두 사용자가 **모두 평가한** 아이템
- 예측 평점: 이웃이 자기 평균보다 얼마나 높게(낮게) 줬는지를 유사도만큼 반영해 내 평균에 더함

$$
\hat{r}_{ui} = \bar{r}_u + \frac{\sum_{v} sim(u, v)\,(r_{vi} - \bar{r}_v)}{\sum_{v} |sim(u, v)|}
$$

- 예시: A는 라라랜드를 좋아할까?
    - $sim(A,B) \approx 0.97$, $sim(A,C) = -1$
    - 평균: A ≈ 3.33, B = 4.25, C = 2.5
    - B는 자기 평균보다 +0.75, C는 −0.5

$$
\hat{r}_{A,\text{라라랜드}} = 3.33 + \frac{0.97 \times 0.75 + (-1) \times (-0.5)}{0.97 + 1} \approx 3.96
$$

    - 약 4점 예측 → **추천**

>> 3) 아이템 기반(Item-based)
- 평점 패턴이 비슷한 **아이템**을 찾아 추천
- 속성이 아닌 **사용자들이 함께 좋아하고 싫어한 패턴**으로 유사도 판단
- 아이템 관계는 상대적으로 덜 변해 유사도를 **미리 계산해 재사용** 가능
- 예: 어바웃 타임과 노팅힐은 평가가 함께 움직임 → 유사 아이템
    - 어바웃 타임을 낮게 준 사용자 C → 노팅힐 비추천
    - 어바웃 타임을 좋아한 사용자 D → 노팅힐 추천

>> 4) User-based vs Item-based
- **User-based**
    - 정확성: 아이템 수가 사용자 수보다 많을 때 유리
    - 공간: 사용자 간 유사도 저장 → 사용자가 많으면 부담
    - 안정성: 사용자 변화 영향이 큼, 아이템이 자주 바뀌고 사용자 유입이 적은 환경에 적합
    - 속도: 추천할 때마다 유사 사용자를 새로 탐색
    - 다양성: 독특한 아이템까지 추천해 **의외성·참신성** 강점
    - 활용: 뉴스, 소셜 미디어(친구 추천), 음악 추천
- **Item-based**
    - 정확성: 사용자 수가 아이템 수보다 많을 때 유리
    - 공간: 아이템 간 유사도 저장 → 아이템이 많으면 부담
    - 안정성: 아이템 목록이 잘 변하지 않는 환경에서 안정적
    - 속도: 유사도를 미리 계산해 **빠름**
    - 다양성: 기존에 본 것과 비슷한 추천이 많아 상대적으로 부족
    - 활용: 아마존 등 사용자가 많은 대형 플랫폼
- 사용자 수가 많으면 아이템 기반이 유리한 이유
    - 두 사용자가 함께 평가한 아이템은 적어도, 두 아이템을 함께 평가한 사용자는 많음
    - 새 사용자가 새 아이템보다 자주 추가되므로 아이템 유사도 갱신은 가끔만 필요
- 일반적 경향: 새로운 추천이 필요한 플랫폼은 User-based, 정확·안정적 추천이 중요한 대형 플랫폼은 Item-based

> ### 메모리 기반 협업 필터링의 한계

>> 1) 희소 행렬 문제
- 평점·선호도 데이터가 적으면 성능 저하
- 사용자-아이템 행렬은 대부분 비어 있음 (**sparse matrix**)
    - 넷플릭스 프라이즈: 약 48만 명 × 1만 8천 편 ≈ 85억 칸, 실제 평점 약 1억 개 → 약 **99% 공백**
- 문제
    - 공통 평가 아이템이 적어 유사도를 믿기 어려움 (공통 1개여도 코사인 유사도 1 가능)
    - 이웃을 못 찾으면 예측 자체가 어려움
- 해결: **모델 기반 협업 필터링**

>> 2) 콜드 스타트 문제
- 행동 기록에만 의존해 기록 없는 대상은 추천 불가
    - **신규 사용자**: 비슷한 이웃을 찾을 수 없음
    - **신규 아이템**: 아무도 평가하지 않아 후보에 오르지 못함 (**first-rater 문제**)
- 해결 방법
    - 가입 시 선호 장르·작품 **직접 지정**
    - 새 아이템에 **내용 기반 필터링**을 함께 쓰는 하이브리드
    - 같은 카테고리 아이템의 **임베딩 평균**으로 임시 표현
    - **인구통계학적 정보**로 유사 사용자 그룹 추정

>> 3) 확장성 문제
- 사용자·아이템이 늘수록 유사도 계산량 증가 (샘플링하면 빨라지지만 품질 하락)
- 해결: **KNN 협업 필터링**
    - 유사도 높은 **k명의 이웃만** 사용
    - 이웃은 예측 대상 아이템을 **실제로 평가한 사용자** 중에서만 선택
    - k가 작으면 소수 의견에 예측이 흔들리고, 크면 덜 비슷한 사용자가 섞여 흐려짐

> ### 모델 기반 협업 필터링

>> 1) 개념
- 관측된 평점으로 **예측 모델을 학습**, 패턴을 파라미터에 담아 미관측 평점 예측
- 메모리 기반: 추천 때마다 저장된 평점을 직접 비교
- 모델 기반: 평점 데이터를 **압축된 모델로 요약**해 두고 추천 수행
- 두 단계
    - **학습 단계**: 평점을 가장 잘 설명하는 파라미터 탐색 (사전 수행, 주기적 재학습)
    - **예측 단계**: 학습된 파라미터로 빈칸 평점 계산 (빠름)
- 등장 배경
    - 메모리 기반은 평점 간 유사도에만 의존 → 숨은 취향 구조 포착 불가, 희소성·확장성에 취약
    - 모델 기반은 **잠재 패턴**을 학습해 직접 겹치는 평가가 없어도 빈칸 추정
- 메모리 vs 모델 기반 (학습 단계·예측 근거·희소 데이터)
    - 메모리 기반: 학습 없음 / 원본 평점 사용 / 공통 아이템이 적어 근거 부족
    - 모델 기반: 학습 있음 / 학습된 모델 사용 / 전체 패턴으로 빈칸 일반화

>> 2) 잠재 요인 모델(Latent Factor Model)
- **잠재 요인**: 평점에 직접 드러나지 않지만 사용자-아이템 관계를 결정하는 **숨은 특성** (취향, 성향 등을 숫자로 표현)
- 사용자와 아이템을 **같은 잠재 공간의 벡터**로 표현
- 평점 행렬을 `사용자-잠재 요인`, `아이템-잠재 요인` 행렬로 나눠 학습
- **Factor Model vs Latent Factor Model**
    - Factor Model: 축의 의미를 사람이 미리 정함 (액션, 로맨스)
    - Latent Factor Model: 축 개수 $k$만 정하고 의미는 **학습으로 결정** → 해석 어려움

>> 3) 행렬 분해(Matrix Factorization)
- 큰 평점 행렬을 **작은 행렬 두 개의 곱**으로 분해

$$
\hat{R} = P^T Q
$$

    - $\hat{R}\,(m \times n)$: 예측 평점 행렬
    - $P\,(k \times m)$: 사용자 잠재 요인 행렬
    - $Q\,(k \times n)$: 아이템 잠재 요인 행렬
    - $k$: 잠재 요인 개수 (하이퍼파라미터)
- 예측 평점은 벡터 **내적**

$$
\hat{r}_{ui} = p_u^T q_i = \sum_{s=1}^{k} p_{su}\, q_{si}
$$

- 예시 (rank-2): 사용자 $p=(0.9, 0.2)$
    - 노팅힐 $q=(0.8, 0.1)$ → 0.74
    - 존 윅 $q=(0.1, 0.9)$ → 0.27
    - 노팅힐 추천
    - 요인 이름은 설명용 임의 표현, 실제 요인은 깔끔하게 해석되지 않는 경우가 대부분
- **학습 과정**
    1. 잠재 요인 개수 $k$ 설정
    2. 모든 벡터를 임의 값으로 초기화
    3. **평점이 있는 칸만** 예측해 오차 계산
    4. 오차가 줄도록 벡터 수정 (SGD, ALS)
    5. 3~4 반복
    6. 학습된 벡터로 **빈칸 예측**
- **손실 함수**

$$
\sum_{\text{평점이 있는 칸}} (\text{실제 평점} - \text{예측 평점})^2 + \lambda \times (\text{벡터 크기})^2
$$

    - 앞부분: 예측 오차 최소화
    - 뒷부분: **규제**, 벡터가 지나치게 커지는 것 방지 ($\lambda$가 클수록 강한 규제)
- **최적화 방법**
    - **SGD**: 평점을 하나씩 보며 오차가 줄어드는 방향으로 수정
    - **ALS**: 사용자 벡터와 아이템 벡터를 번갈아 고정하고 나머지를 계산, 병렬 처리 유리
- **편향(Bias)**

$$
\hat{r}_{ui} = \mu + b_i + b_u + p_u^T q_i
$$

    - $\mu$: 전체 평균 평점
    - $b_i$: 아이템 편향 (평균보다 높게·낮게 평가받는 정도)
    - $b_u$: 사용자 편향 (평균보다 높게·낮게 평가하는 정도)
    - 예: 평균 3.7, 타이타닉 +0.5, 깐깐한 Joe −0.3 → 기본 예측 **3.9점**
    - 편향이 경향을 먼저 설명하면 잠재 요인은 **사용자-아이템 궁합**에 집중
- **행렬 분해 방식**
    - **SVD**: 수학적 분해, 희소 행렬에 직접 적용 불가해 결측치 처리 필요
    - **학습 기반 MF**: SGD·ALS로 관측된 평점만 학습해 희소성 우회
    - **NMF**: 비음수 제약으로 잠재 요인 해석력 향상
    - **PMF**: 평점이 내적을 평균으로 하는 정규분포를 따른다고 보는 확률 모델

>> 4) 메모리 기반과의 차이
- **메모리 기반**
    - 예측: 비슷한 사용자·아이템을 직접 탐색
    - 확장성: 추천마다 많은 유사도 계산
    - 콜드 스타트: 해결 어려움
    - 장점: 추천 이유 설명 용이
- **모델 기반**
    - 예측: 학습한 벡터를 내적
    - 확장성: 학습은 사전 수행, 추천 시 내적만 계산해 빠름
    - 콜드 스타트: 해결 어려움 (상호작용이 없으면 벡터 학습 불가)
    - 장점: 데이터 압축, 빠른 예측, 일반화 유리
- 한계: 희소성·확장성은 많이 해결했지만 **내적이라는 선형 계산**의 한계가 남음 → 딥러닝으로 극복

---

## VAE 기반 협업 필터링

> ### MF의 한계와 VAE 구조

>> 1) MF의 한계
- **선형성**
    - 예측이 내적, 즉 요인별 곱의 합이라 요인 간 상호작용 표현 어려움
    - 예: 로맨스·음악 영화는 좋아하지만 뮤지컬 로맨스는 싫은 사용자 → 내적에서는 둘 다 높은 영화가 오히려 최고점
- **불확실성 무시**
    - 사용자마다 벡터 하나만 학습(**점 추정**)
    - 기록 5개 사용자와 5,000개 사용자를 똑같이 확신
- **딥러닝의 해법**
    - 선형성 → **비선형 활성화 함수를 가진 신경망**
    - 불확실성 → **평균과 분산을 함께 출력**

>> 2) 오토인코더(AE)와 VAE
- **AE**: 입력을 작은 벡터로 압축했다가 복원하는 신경망
    - `입력 x → 인코더 → 잠재 벡터 z → 디코더 → 복원 x̂`
    - 핵심 정보만 남기도록 학습, **입력 자체가 정답**
    - 초기 시도 **AutoRec**: 아이템 평점 벡터를 복원
        - RMSE: 선형 0.872 → sigmoid 0.831 (낮을수록 좋음)
        - 신경망 **비선형성**이 성능을 높임을 보여 줌
    - 한계: 사용자를 **점 하나**로 표현 (MF와 동일)
- **VAE**: 잠재 벡터를 **분포**로 표현하는 오토인코더
    - **평균 μ**: 취향의 중심 위치
    - **표준편차 σ**: 위치에 대한 불확실성
- 예시: 본 영화 5편(로맨스 3, SF 1, 공포 1)
    - AE: z = `[1.2, -0.7]` 로 확정 → 우연히 본 공포 영화 같은 노이즈까지 반영
    - VAE: μ = `[1.2, -0.7]`, σ = `[0.8, 0.9]` → 주변 z를 샘플링해 비슷한 위치에서 비슷한 추천
- VAE가 더 안정적인 이유
    - 점이 아닌 **주변까지 사용해 학습**
    - KL 항이 **기준 분포(표준 정규분포) 쪽으로 당겨** 극단적 위치 방지

>> 3) Mult-VAE 구조
- **입력 $x$**: 아이템 수만큼 긴 0/1 벡터 (평점은 4점 이상만 1)
- **인코더**: 평균 μ, 표준편차 σ 출력
- **잠재 벡터 $z$**: μ, σ의 정규분포에서 **샘플링**
- **디코더**: 모든 아이템 점수 계산 후 **softmax**로 확률 변환
- 확률 높은 순으로, 아직 상호작용하지 않은 아이템 추천

> ### 목적 함수와 KL Divergence

>> 1) ELBO (Evidence Lower BOund)
- 데이터가 나올 로그 확률의 **하한**을 키우는 목적 함수
- **정확한 복원**과 **잠재 공간 정규화**를 동시에 최적화

$$
\mathcal{L}(x; \theta, \phi) = \underbrace{E_{q_\phi(z|x)} [ \log p_\theta(x|z) ]}_{\text{복원 항}} - \underbrace{D_{\text{KL}}(q_\phi(z|x) \,\|\, p(z))}_{\text{KL Divergence 항}}
$$

- **복원 항**: 디코더가 $z$로 원래 입력 $x$를 최대한 비슷하게 복원 (AE 역할)
- **KL 항**: $q(z|x)$가 표준 정규분포 $p(z)$에서 크게 벗어나지 않도록 강제
    - 과적합 방지, 잠재 공간을 **연속적이고 부드럽게** 유지
- 비유: 복원 항은 "기출 문제 많이 풀기", KL 항은 "기본 개념에서 벗어나지 않기"
    - 복원 항만 있으면 기출 답만 외워 새 문제를 못 풂
    - KL 항만 있으면 모든 문제에 같은 답

>> 2) KL Divergence
- **두 확률분포가 얼마나 다른지** 나타내는 값
- 두 분포가 모두 정규분포면 식으로 직접 계산

$$
D_{KL}\big(\mathcal{N}(\mu, \sigma^2) \,\|\, \mathcal{N}(0, 1)\big) = \frac{1}{2}\left(\mu^2 + \sigma^2 - \ln \sigma^2 - 1\right)
$$

- 값 예시
    - μ=0, σ=1: **0** (벌점 없음)
    - μ=2, σ=1: **2.0** (중심에서 멀음)
    - μ=0, σ=0.1: **약 1.81** (너무 좁게 확신)
    - μ=0, σ=3: **약 2.90** (너무 넓게 퍼짐)
- **중심에서 멀어지거나, 지나치게 확신하거나, 지나치게 퍼지면** 벌점 증가

>> 3) AE vs VAE의 잠재 공간
- **AE**
    - 복원만 정확하면 OK → 벡터가 제약 없이 흩어짐, 빈 공간은 학습되지 않음
    - 기록이 적은 사용자는 우연한 기록(노이즈) 때문에 엉뚱한 곳에 놓이기 쉬움
- **VAE**
    - 분포가 N(0, 1) 쪽으로 당겨져 중심 근처에 **연속적으로** 모임
    - 기록이 적은 사용자는 근거가 부족한 만큼 중심 쪽으로 당겨져 비슷한 사용자 근처에 안정적으로 위치
    - 기록 많은 사용자에게서 학습된 패턴을 함께 활용 → **희소한 데이터에 강함**

>> 4) VAE-CF의 주요 장점
- **확률적 잠재 분포 모델링**: 불확실성 반영, 노이즈 영향 감소
- **희소 데이터에서 안정적**: KL 정규화로 희소성 문제에 효과적 대응
- **비선형성 학습**: MF의 한계였던 복잡한 비선형 패턴 학습

---

## GNN 기반 사용자 선호 예측

> ### GNN 구조와 프로세스

>> 1) 등장 배경
- CF, MF, VAE는 상호작용을 **행렬의 칸 하나하나**로 다룸
    - "A가 본 영화를 B도 봤고, B는 Y도 봤다" 같은 **연결된 관계**는 칸만으로 활용 불가
    - 행동의 **순서**도 고려하지 않음
- 사용자-아이템 상호작용은 본질적으로 **그래프**
    - 사용자·아이템 = 점, "봤다·샀다" = 선
    - 경로 예: `나 → 영화 X → (X를 본) 다른 사용자 → 영화 Y`

>> 2) 그래프(Graph)
- **노드(Node)**와 **엣지(Edge)**로 구성된 자료 구조
    - 노드: 개체 (사용자, 아이템, 원자 등)
    - 엣지: 노드 간 관계·상호작용 (방향, 가중치 가능)
    - 예: 소셜 네트워크 — 노드=사용자, 엣지=친구·팔로우, 가중치=메시지 횟수
- **그래프를 쓰는 이유**
    - **관계·상호작용** 같은 추상 개념을 `[노드 - 엣지 - 노드]`로 저장해 관계 자체를 직접 학습
    - **Non-Euclidean space** 데이터 표현·학습 가능
        - 이미지: 격자 구조, 이웃 순서 고정 → CNN
        - 텍스트: 일렬 시퀀스, 앞뒤 이웃 고정 → RNN
        - SNS·분자·추천: 노드마다 이웃 수 제각각, 정해진 좌표·순서 없음 → GNN
        - 행렬·벡터로 억지로 펼치면 정보 손실·비효율 발생
        - 노드·엣지 구조를 그대로 쓰는 GNN으로 학습

>> 3) GNN 작동 구조
- 핵심 아이디어: **이웃 노드의 정보를 나에게 전달·요약(Message Passing)**
- 단계
    1. **그래프 구조 생성**: 노드·엣지 정의, 초기 피처(임베딩) 부여
    2. **메시지 패싱**
        - **메시지 생성**: 이웃 노드가 자신의 임베딩으로 메시지 생성
        - **집계**: 받은 메시지를 하나로 합침 (평균, 합, 최대값, attention)
        - **갱신**: 집계 결과와 자신의 이전 임베딩을 결합해 새 임베딩 생성
    3. **최종 출력**
- $K$번 반복하면 $K$단계 떨어진 이웃 정보까지 반영
- 임베딩을 갱신하는 대상 = **타깃 노드** (모든 노드가 한 번씩 담당)

>> 4) 그래프 구성 예시 (Netflix)
- 사용자 3(U1~U3) + 영화 3(A~C) = 노드 6개, 평점 기록 5개 = 엣지 5개
- 연결: `B — U1 — A — U2 — C — U3`
- 행렬 표현
    - **인접 행렬**: 노드 간 관계 (6×6)
    - **피처 행렬**: 노드 고유 feature (예: 20대, 30대, 액션, SF, 평균평점 4.5 이상)
- 메시지 패싱 예 (U1이 타깃)
    - 메시지 생성: 영화 A, B가 각자 임베딩으로 메시지 생성
    - 집계: A, B의 메시지를 평균해 하나의 벡터로 요약
    - 갱신: U1의 이전 임베딩 + 요약 벡터 결합
- 수식
    - 메시지: $m_u^{(l)} = MSG^{(l)}(h_u^{(l-1)})$
    - 집계: $h_v^{(l)} = AGG^{(l)}(\{m_u^{(l)} \mid u \in N(v)\})$
    - 갱신: $h_v^{(l)} = Combine(AGG^{(l)}(\cdot), m_v^{(l)})$, $h_v^{(l+1)} = \sigma(W^{(l)} \cdot h_v^{(l)})$

> ### LightGCN

>> 1) 주요 GNN 아키텍처
- 아키텍처 차이는 **메시지를 어떻게 만드는가**, **어떻게 합치는가**로 결정
- **GCN**: 그래프 위 Convolution, 이웃·타깃에 동일 파라미터 적용
    - 이웃 임베딩을 연결 수(차수)로 정규화해 평균, 학습 시 전체 그래프 필요
- **GRN**: RNN + GNN 결합
    - 시간에 따라 변하는 동적 관계 포착
- **GAT**: 이웃 중요도를 attention으로 학습해 가중 평균
    - 중요한 이웃에 집중하지만 계산량 증가
- **GraphSAGE**: 이웃을 일부만 샘플링, 이웃 정보와 자기 정보를 더하지 않고 **concat**
    - 대규모 그래프와 **학습 때 없던 새 노드**에도 적용 가능
- 비유 (카페 신메뉴, 민수의 주문 예측)
    - GNN: 친구 5명 의견을 똑같이 평균
    - GAT: 취향이 비슷한 친구 의견에 더 큰 비중
    - 학습으로 바뀌는 것은 $W$(와 attention 파라미터), 엣지·초기 노드 특징은 고정
- **GNN**: 그래프 데이터를 학습하는 신경망 전체 개념 / **GCN**: 합성곱으로 이웃 정보를 집계하는 구체적 모델
- 추천에서는 GCN 계열이 가장 널리 사용: **NGCF**(2019) → 단순화한 **LightGCN**(2020)

>> 2) LightGCN
- **탄생 배경**
    - NGCF는 레이어마다 **가중치 행렬 $W$**와 **비선형 활성화 함수** 사용
    - 이는 풍부한 피처가 있을 때 유용하나, 추천 그래프의 노드는 **ID 임베딩**뿐
    - 실험 결과 두 연산은 도움이 되지 않고 오히려 학습을 방해
- **제거한 구성 요소**
    - **특징 변환 제거**: 레이어별 $W^{(l)}$ 삭제
    - **비선형 활성화 함수 제거**: ReLU 없이 정규화된 합·평균으로 이웃 정보 결합
        - 이유: 임베딩 전파 과정에서 비선형성이 충분히 반영, 과적합 방지
- 구성: **정규화된 이웃 임베딩의 합 + 레이어 결합**
- 같은 조건에서 NGCF보다 **평균 약 16% 높은 성능** (He et al., SIGIR 2020)

> ### GNN 활용 사례 및 장단점

>> 1) 활용 사례
- **Pinterest**
    - 핀과 보드를 노드, '핀이 보드에 저장됨'을 엣지로 구성
    - 약 30억 노드, 180억 엣지, 75억 학습 예시
    - 전체 그래프 연산이 불가능해 **GraphSAGE**처럼 중요한 이웃만 샘플링
- 그 외 분야
    - **소셜 네트워크**: 관계 경로 분류, 친구 추천
    - **추천 시스템**: 사용자-아이템 관계 분석
    - **화학·생명과학**: 분자 구조·특성 예측, 약물 상호작용 예측 (노드=원자, 엣지=결합)
    - **금융·보안**: 이상 거래 탐지, 악성 사용자 분류
    - **Traffic**: 장소 간 소요 시간 예측
    - **프로그램 분석**: 코드 auto suggestion

>> 2) 장점
- **고차원 협업 시그널** 포착
    - 이웃 사용자와 연관 아이템의 정보를 반복 반영
- **정형화되지 않은 데이터 학습**
    - 상호작용, 소셜 관계, 지식 그래프, 사용자 속성을 하나의 그래프로 학습
- **Cold Start 완화**
    - 상호작용 없는 신규 아이템도 속성 노드·이웃 정보로 임베딩 획득

>> 3) 단점
- **과평탄화(Over-smoothing)**
    - 레이어가 많아질수록 모든 노드 임베딩이 같은 값으로 수렴
- **과압축(Oversquashing)**
    - 기하급수적으로 늘어나는 이웃 정보를 고정 크기 벡터에 담아 먼 노드 정보 손실
- **계산 복잡도 증가**
    - 수십억 노드·엣지에 막대한 자원·메모리 필요
- **순서 반영의 어려움**
    - 상호작용을 집합으로 봐 '신발→바지→셔츠'와 '셔츠→바지→신발'이 같은 그래프
    - 다음 행동 예측 곤란

---

## Transformer 기반 추천 시스템

> ### 기존 추천 시스템의 한계

>> 1) CF의 순차적 한계
- **순서 무시**: 구매 순서가 달라도 같은 사용자로 간주
- **문맥 부족**: '방금' 구매와 '1년 전' 구매의 중요도 구분 불가
- **예측의 한계**: '다음' 행동보다 '일반적' 선호 예측에 초점

>> 2) GNN의 순차적 한계
- **지역적 정보 전파의 한계**
    - `[A → B → C]`에서 C가 A의 정보를 받으려면 B를 거쳐야 함 (레이어 2개 필요)
    - 거치는 동안 정보 희석·압축 (과압축)
- **장기 의존성 문제**
    - 레이어 수만큼의 단계까지만 확인 가능
    - Self-Attention은 시퀀스 내 모든 아이템 쌍을 거리와 무관하게 한 번에 비교
- 정리: CF는 '누가 무엇을 좋아하는가', GNN은 '무엇이 무엇과 연결되었는가'를 포착하지만 **'어떤 순서로, 어떤 의도로'** 행동하는지는 못 봄
    - 시퀀스 전체를 한 번에 보며 현재 맥락에 중요한 과거 행동에 집중하는 **Transformer**가 대안

> ### Transformer와 Transformer 기반 주요 추천 시스템 모델

>> 1) 사용자 행동을 시퀀스로 접근
- 최근 행동 시퀀스에 현재 세션의 목적이 반영 → 다음 행동은 평균 취향보다 **최근 행동의 순서·조합(문맥)** 영향이 큼
- **문맥에 따른 동일 아이템의 의미 (Self-Attention)**
    - 수영복 → 비치타월 → **선크림**: 휴가 준비물 → 방수팩, 래시가드 추천
    - 스킨 → 로션 → **선크림**: 데일리 스킨케어 → 클렌징폼, 선크림 리필 추천
- **시간 순서를 넘는 핵심 의도 포착 (Long-Range Dependency)**
    - **프린터** → 양말 → 라면 → 폰케이스 → 생수 → 샴푸 → (?)
    - 관련 없는 행동 5개 뒤에도 프린터에 높은 가중치 → **토너** 추천
    - RNN·GNN은 단계를 거치며 정보가 희석되지만 Transformer는 직접 반영

>> 2) Self-Attention
- 시퀀스 내 모든 아이템 쌍의 관련성을 거리와 상관없이 **동시에 계산**
- **Query / Key / Value**
    - **Query(Q)**: 아이템 임베딩 × $W_Q$, 다른 아이템과의 관련성을 계산하는 기준
    - **Key(K)**: 아이템 임베딩 × $W_K$, Query와 내적해 관련성 점수 결정
    - **Value(V)**: 아이템 임베딩 × $W_V$, 점수에 따라 가중합되는 실제 정보

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d}}\right)V
$$

- 연산 과정 (선크림 기준)
    1. **Attention Score** (Q·K): 수영복 2.0 / 비치타월 1.5 / 양말 0.1
    2. **Softmax**: 수영복 57.0% / 비치타월 34.5% / 양말 8.5%
    3. **가중합**: `0.570 × V(수영복) + 0.345 × V(비치타월) + 0.085 × V(양말)`
- 수영복·비치타월 정보가 약 92% → '휴가 준비물'로 해석
- 핵심: **관련성에 따라 정보 반영 비율을 다르게 적용**

>> 3) SASRec
- **단방향 어텐션**으로 과거 정보만 활용해 **다음 아이템 예측** (GPT와 유사)
- 구조
    - 입력: 시간순 행동 시퀀스, 각 위치의 정답은 바로 다음 아이템
    - Embedding Layer: 아이템 임베딩 + 위치 임베딩
    - Self Attention Layer: 과거 아이템만 참조
    - FFN: 위치별 비선형 변환
    - Prediction Layer: 출력 벡터와 아이템 임베딩 내적으로 다음 아이템 점수 계산
- 예: `[노트북 → 마우스 → 키보드]` → '노트북 거치대'
- 특징
    - **적응적 집중**: 데이터가 희소하면 최근 아이템, 풍부하면 긴 과거 맥락에 집중
    - **빠른 학습**: 모든 위치를 병렬 처리
- 문제점
    - **계산량**: Attention 연산량이 시퀀스 길이의 제곱($n^2$)에 비례
    - **단방향성**: 앞부분 아이템은 활용할 정보가 적고, 장바구니 동시 담기처럼 순서가 엄격하지 않은 경우 존재

>> 4) BERT4Rec
- **양방향 어텐션**으로 과거·미래 정보를 모두 활용해 **가려진 아이템 예측** (BERT와 유사)
- 구조
    - 입력: 일부 아이템을 `[MASK]`로 가림
    - Embedding Layer: 아이템 임베딩(v) + 위치 임베딩(p)
    - `Trm × L`: 양방향 Self-Attention을 L층 쌓은 구조
    - Projection: `[MASK]` 위치 출력으로 원래 아이템 예측
- `[MASK]`를 쓰는 이유: 양방향 학습 시 target 아이템을 직접 보는 **정보 누출(information leakage)** 방지
- 예: `[노트북 → 마우스 → [MASK] → 키보드]` → '노트북 거치대' 예측
- 특징
    - **양방향 문맥**: 앞뒤 문맥을 모두 학습에 사용
    - **데이터 활용**: 마스킹 위치를 바꿔 한 시퀀스에서 여러 샘플 생성

>> 5) SASRec vs BERT4Rec
- **SASRec**
    - 어텐션 방향: 단방향 (과거만)
    - 학습 목표: 다음 아이템 예측
    - 학습 효율: 빠름
- **BERT4Rec**
    - 어텐션 방향: 양방향 (과거 + 미래)
    - 학습 목표: 가려진 아이템 예측
    - 학습 효율: 상대적으로 느림
- **BERT4Rec이 항상 더 좋은가?**
    - 원 논문(Sun et al., CIKM 2019): BERT4Rec이 SASRec보다 높은 성능
    - 후속 연구(Klenitskiy & Vasilev, RecSys 2023): 손실 함수를 같게 맞추자 SASRec이 비슷하거나 더 높고 학습도 빠름
        - Beauty 데이터셋 NDCG@10: 개선된 SASRec 0.0327 vs BERT4Rec 0.0187
    - 성능 차이는 어텐션 방향보다 **학습 설정**의 영향이 큼
- **상황별 선택 기준**
    - 상호작용이 많고 '다음 행동' 예측이 목적 (실시간 피드, 커머스) → **SASRec** (효율 높고 손실 함수를 맞추면 성능 동등 이상)
    - 시퀀스가 짧은 경우 → 상황에 따라 다름
        - BERT4Rec은 마스킹으로 데이터 증강 효과가 있으나, 손실 함수를 맞추면 차이가 줄어든다는 반론도 있음
    - 행동 순서가 느슨한 도메인 (장바구니 동시 담기) → **BERT4Rec** (양방향 문맥이 의미 있음)

---

## 오늘날의 추천 시스템

> ### LLM의 시대

>> 1) 관점의 전환
- 추천을 **랭킹 문제가 아닌 언어 문제**로 해결
    - 과거: (사용자 A, 아이템 123) → 0.8점 같은 점수 출력
    - LLM: 문장으로 출력, ID가 아닌 **의미(Semantic)** 기반, 멀티모달 데이터 활용
- 점수 대신 추천 이유를 함께 제시하는 **서술형 추천** 가능

>> 2) 새로운 가능성
- **대화형 추천**: 자연어 조건 반영
    - 예: "비 오는 일요일에 혼자 보기 좋은, 너무 슬프지 않은 영화"
- **Zero-shot 추천**: 상호작용 데이터 없이 텍스트 설명만으로 추천
    - 신상품 첫날 구매 기록 0건이어도 추천 가능 → **Cold-Start**의 강력한 해결책

>> 3) 대표 모델
- **P5** (Geng et al., RecSys 2022)
    - 여러 추천 과제를 "텍스트 프롬프트 → 텍스트 답변"으로 통일해 하나의 언어 모델로 학습
    - 단점: 추천 데이터로 처음부터 학습해 컴퓨팅 자원·시간 부담
- **TALLRec** (Bao et al., RecSys 2023)
    - 범용 LLM을 "좋아할까? (Yes/No)" 형식 지시문으로 튜닝
    - **LoRA**로 일부 파라미터만 학습, 적은 샘플로도 기존 모델보다 높은 성능 보고

> ### 인기 편향과 XAI

>> 1) 인기 편향(Popularity Bias)
- 소수 인기 아이템이 추천을 독점하고 비인기 아이템은 거의 추천되지 않는 현상
- 강화 3단계
    1. **데이터 불균형**: 상호작용이 인기 아이템에 집중
    2. **모델의 편향 강화**: 데이터 많은 아이템일수록 점수가 높아 더 자주 추천
    3. **Rich-get-richer**: 추천 → 노출 → 클릭 → 학습 반영 반복으로 격차 확대
- 사례
    - Netflix Prize: 인기 영화 16.5%(2,938편)가 전체 평점의 약 86%
    - Spotify 팟캐스트: 스트리밍 28.90% 증가, 개인 소비 다양성 11.51% 감소
- 결과: 비인기·신규 아이템은 노출 기회가 적고, 사용자는 비슷한 콘텐츠만 반복 추천

>> 2) 설명가능성(XAI)
- 문제: GNN, Transformer 등 딥러닝은 **블랙박스**
- 필요성: 고객 설득, 비즈니스 담당자에게 근거 제시
- LLM은 추천 이유를 자연어로 생성할 수 있으나 **실제 추천 근거와 일치한다는 보장은 없음**

> ### 컴퓨팅 비용과 2단계 구조

>> 1) 비용 문제
- **GPU 비용**: 대형 모델 학습·추론에 고가 GPU 필요
- **실시간 스코어링 서버비**: 요청마다 전체 아이템에 딥러닝 연산 시 급증
- **지속 비용**: 사용자 수 × 요청 수만큼 매일 누적
- 선형 모델은 정확도는 낮지만 빠르고 저렴하며 해석 쉬움, 딥러닝·LLM은 정확도가 높지만 무거움

>> 2) 2단계 구조 (유튜브, 아마존 등)
- **1단계: 후보군 생성**
    - 목표: 수백만 개 아이템 → 수백 개 후보
    - 모델: **선형 모델** (내용 기반, 협업 필터링, 행렬 분해)
    - 장점: 빠른 처리, 낮은 학습 비용
- **2단계: 순위 매기기**
    - 목표: 수백 개 후보를 정교하게 정렬해 상위 수십 개 노출
    - 모델: **딥러닝 모델** (GNN, Transformer, VAE)
    - 장점: 비선형성·순서·관계 등 문맥을 총동원해 높은 정확도, 선형 모델이 놓치는 패턴 포착
- 결론: 실제 추천 시스템은 단일 모델이 아니라 **모델을 단계별로 조합**해 비용과 정확도를 함께 확보